In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from numpy import NaN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import glob
import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

2023-02-10 13:01:55.001006: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-02-10 13:01:55.116023: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-02-10 13:01:55.651146: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-02-10 13:01:55.651194: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] 

In [2]:
# Saving all .csv files in folder to list.
path = "MachineLearningCVE/"
files = [file for file in glob.glob(path + "**/*.csv", recursive=True)]

In [3]:
[print(f) for f in files]

MachineLearningCVE/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv
MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv
MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv
MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv


[None, None, None, None, None, None, None, None]

In [4]:
# Reading all the csv files into dataframes and putting thoose DFs to one list.

dataset = [pd.read_csv(f) for f in files]

In [5]:
# shape of the each files

for d in dataset:
    print(d.shape)

(286467, 79)
(288602, 79)
(692703, 79)
(445909, 79)
(225745, 79)
(191033, 79)
(170366, 79)
(529918, 79)


In [6]:
#comparing the columns of each tables

for i in range(0,len(dataset)):
    if i != len(dataset)-1:
        same_columns = dataset[i].columns == dataset[i+1].columns
        
        if False in same_columns:
            print(i)
            break

same_columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True])

In [7]:
 #Combining all tables into one dataset

dataset = pd.concat([d for d in dataset]).drop_duplicates(keep=False)
dataset.reset_index(drop=True, inplace = True)

In [8]:
# Saving combined data

dataset.to_csv("Dataset_work.csv", index=False)

In [9]:
# dataset shape

dataset.shape


(2427193, 79)

In [10]:
# checking the datatypes of each features
#dataset.dtypes
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2427193 entries, 0 to 2427192
Data columns (total 79 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0    Destination Port             int64  
 1    Flow Duration                int64  
 2    Total Fwd Packets            int64  
 3    Total Backward Packets       int64  
 4   Total Length of Fwd Packets   int64  
 5    Total Length of Bwd Packets  int64  
 6    Fwd Packet Length Max        int64  
 7    Fwd Packet Length Min        int64  
 8    Fwd Packet Length Mean       float64
 9    Fwd Packet Length Std        float64
 10  Bwd Packet Length Max         int64  
 11   Bwd Packet Length Min        int64  
 12   Bwd Packet Length Mean       float64
 13   Bwd Packet Length Std        float64
 14  Flow Bytes/s                  float64
 15   Flow Packets/s               float64
 16   Flow IAT Mean                float64
 17   Flow IAT Std                 float64
 18   Flow IAT Max         

In [11]:
# Removing whitespaces in column names.

col_names = [col.replace(' ', '') for col in dataset.columns]
dataset.columns = col_names
dataset.head()

,DestinationPort,FlowDuration,TotalFwdPackets,TotalBackwardPackets,TotalLengthofFwdPackets,TotalLengthofBwdPackets,FwdPacketLengthMax,FwdPacketLengthMin,FwdPacketLengthMean,FwdPacketLengthStd,...,min_seg_size_forward,ActiveMean,ActiveStd,ActiveMax,ActiveMin,IdleMean,IdleStd,IdleMax,IdleMin,Label
0,22,1266342,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,22,1319353,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,22,1303488,41,42,2728,6634,456,0,66.536585,110.129945,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,35396,77,1,2,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,22,1307239,41,40,2728,6634,456,0,66.536585,110.129945,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [12]:
# Dataset conatains 15 labels.
#print(dataset[' Label'].unique())
#len(dataset[' Label'].unique())

dataset['Label'].value_counts()

BENIGN                        2036840
DoS Hulk                       171509
DDoS                           128007
PortScan                        57429
DoS GoldenEye                   10279
FTP-Patator                      5481
DoS slowloris                    5289
DoS Slowhttptest                 5176
SSH-Patator                      3071
Bot                              1947
Web Attack � Brute Force         1445
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: Label, dtype: int64

In [13]:
# This next snippet uses regular expressions to replace wierd characters with dunders.

label_names = dataset['Label'].unique()


import re

label_names = [re.sub("[^a-zA-Z ]+", "", l) for l in label_names]
label_names = [re.sub("[\s\s]", '_', l) for l in label_names]
label_names = [lab.replace("__", "_") for lab in label_names]

label_names, len(label_names)

(['BENIGN',
  'PortScan',
  'Infiltration',
  'DoS_slowloris',
  'DoS_Slowhttptest',
  'DoS_Hulk',
  'DoS_GoldenEye',
  'Heartbleed',
  'FTPPatator',
  'SSHPatator',
  'DDoS',
  'Bot',
  'Web_Attack_Brute_Force',
  'Web_Attack_XSS',
  'Web_Attack_Sql_Injection'],
 15)

In [14]:
# Replacing 'Label' column values with new readable values.

labels = dataset['Label'].unique()

for i in range(0,len(label_names)):
    dataset['Label'] = dataset['Label'].replace({labels[i] : label_names[i]})
    
dataset['Label'].unique()

array(['BENIGN', 'PortScan', 'Infiltration', 'DoS_slowloris',
       'DoS_Slowhttptest', 'DoS_Hulk', 'DoS_GoldenEye', 'Heartbleed',
       'FTPPatator', 'SSHPatator', 'DDoS', 'Bot',
       'Web_Attack_Brute_Force', 'Web_Attack_XSS',
       'Web_Attack_Sql_Injection'], dtype=object)

In [15]:
# changing attack labels to their respective attack class
# total 38 different sub-attacks 
def change_label(dataset):
    dataset.Label.replace(['DoS_slowloris','DoS_Slowhttptest','DoS_Hulk','DoS_GoldenEye','DDoS'],'Dos',inplace=True)
    dataset.Label.replace(['Web_Attack_Brute_Force','Web_Attack_XSS', 'Web_Attack_Sql_Injection','Bot'],'Botnet',inplace=True)
    dataset.Label.replace(['Heartbleed','PortScan'],'PortScan',inplace=True)
    dataset.Label.replace(['Infiltration','FTPPatator','SSHPatator'],'Firewall',inplace=True)


In [16]:
# calling change_label() function
change_label(dataset)

In [17]:
# distribution of attack classes
dataset.Label.value_counts()

BENIGN      2036840
Dos          320260
PortScan      57440
Firewall       8588
Botnet         4065
Name: Label, dtype: int64

In [18]:
# checking null values 

dataset.isnull().values.any()

True

In [19]:
 #Checking which column/s contain NULL values.

[col for col in dataset if dataset[col].isnull().values.any()]


['FlowBytes/s']

In [20]:
# Checking how many NULL values it this column contains.

dataset['FlowBytes/s'].isnull().sum()

334

In [21]:
# Removing rows that contain NULL values

dataset.dropna(inplace=True)

In [22]:
dataset.isnull().any().any()

False

In [23]:
# converting string values to float64 

labl = dataset['Label']
dataset = dataset.loc[:, dataset.columns != 'Label'].astype('float64')

In [24]:


# Checking if all values are finite.

np.all(np.isfinite(dataset))



False

In [25]:


# Checking what column/s contain non-finite values.

nonfinite = [col for col in dataset if not np.all(np.isfinite(dataset[col]))]

nonfinite



['FlowBytes/s', 'FlowPackets/s']

In [26]:
# Checking how many non-finite values each column contains.

finite = np.isfinite(dataset['FlowBytes/s']).sum()

dataset.shape[0] - finite

1132

In [27]:
# Replacing infinite values with NaN values.
dataset = dataset.replace([np.inf, -np.inf], np.nan)

In [28]:
np.any(np.isnan(dataset))

True

In [29]:
dataset = dataset.merge(labl, how='outer', left_index=True, right_index=True)

In [30]:
dataset.dropna(inplace=True)

In [31]:
dataset.shape

(2425727, 79)

In [32]:
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

In [33]:
# selecting numeric attributes columns from data
numeric_col = dataset.select_dtypes(include='number').columns

# using standard scaler for normalizing
std_scaler = StandardScaler()
def normalization(df,col):
  for i in col:
    arr = df[i]
    arr = np.array(arr)
    df[i] = std_scaler.fit_transform(arr.reshape(len(arr),1))
  return df


In [34]:
# calling the normalization() function
dataset = normalization(dataset.copy(),numeric_col)

# data after normalization
dataset.head()

,DestinationPort,FlowDuration,TotalFwdPackets,TotalBackwardPackets,TotalLengthofFwdPackets,TotalLengthofBwdPackets,FwdPacketLengthMax,FwdPacketLengthMin,FwdPacketLengthMean,FwdPacketLengthStd,...,min_seg_size_forward,ActiveMean,ActiveStd,ActiveMax,ActiveMin,IdleMean,IdleStd,IdleMax,IdleMin,Label
0,-0.459864,-0.446539,0.037514,0.029720,0.188070,-0.004867,0.281243,-0.31171,-0.001137,0.097695,...,0.002761,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN
1,-0.459864,-0.445056,0.037514,0.029720,0.188070,-0.004867,0.281243,-0.31171,-0.001137,0.097695,...,0.002761,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN
2,-0.459864,-0.445500,0.037514,0.027864,0.193999,-0.004998,0.281243,-0.31171,0.006706,0.098573,...,0.002761,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN
3,1.378070,-0.481952,-0.011879,-0.009261,-0.058755,-0.007712,-0.311085,-0.31171,-0.327627,-0.265803,...,0.002761,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN
4,-0.459864,-0.445395,0.037514,0.026008,0.193999,-0.004998,0.281243,-0.31171,0.006706,0.098573,...,0.002761,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN


In [35]:
# multiclass classification ##

In [36]:
multi_data = dataset.copy()
multi_label = pd.DataFrame(multi_data.Label)

In [37]:
from sklearn.preprocessing import LabelEncoder
LE2 = LabelEncoder()

enc_label = multi_label.apply(LE2.fit_transform)
multi_data['intrusion']= enc_label

In [38]:
LE2.classes_

array(['BENIGN', 'Botnet', 'Dos', 'Firewall', 'PortScan'], dtype=object)

In [39]:
multi_data.head()

,DestinationPort,FlowDuration,TotalFwdPackets,TotalBackwardPackets,TotalLengthofFwdPackets,TotalLengthofBwdPackets,FwdPacketLengthMax,FwdPacketLengthMin,FwdPacketLengthMean,FwdPacketLengthStd,...,ActiveMean,ActiveStd,ActiveMax,ActiveMin,IdleMean,IdleStd,IdleMax,IdleMin,Label,intrusion
0,-0.459864,-0.446539,0.037514,0.029720,0.188070,-0.004867,0.281243,-0.31171,-0.001137,0.097695,...,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN,0
1,-0.459864,-0.445056,0.037514,0.029720,0.188070,-0.004867,0.281243,-0.31171,-0.001137,0.097695,...,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN,0
2,-0.459864,-0.445500,0.037514,0.027864,0.193999,-0.004998,0.281243,-0.31171,0.006706,0.098573,...,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN,0
3,1.378070,-0.481952,-0.011879,-0.009261,-0.058755,-0.007712,-0.311085,-0.31171,-0.327627,-0.265803,...,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN,0
4,-0.459864,-0.445395,0.037514,0.026008,0.193999,-0.004998,0.281243,-0.31171,0.006706,0.098573,...,-0.136006,-0.113061,-0.161613,-0.109217,-0.384114,-0.118364,-0.389626,-0.369714,BENIGN,0


In [40]:
# one-hot-encoding for attack label

multi_data = pd.get_dummies(multi_data,columns=['Label'],prefix="",prefix_sep="")
multi_data['Label']= multi_label
multi_data

,DestinationPort,FlowDuration,TotalFwdPackets,TotalBackwardPackets,TotalLengthofFwdPackets,TotalLengthofBwdPackets,FwdPacketLengthMax,FwdPacketLengthMin,FwdPacketLengthMean,FwdPacketLengthStd,...,IdleStd,IdleMax,IdleMin,intrusion,BENIGN,Botnet,Dos,Firewall,PortScan,Label
0,-0.459864,-0.446539,0.037514,0.029720,0.188070,-0.004867,0.281243,-0.311710,-0.001137,0.097695,...,-0.118364,-0.389626,-0.369714,0,1,0,0,0,0,BENIGN
1,-0.459864,-0.445056,0.037514,0.029720,0.188070,-0.004867,0.281243,-0.311710,-0.001137,0.097695,...,-0.118364,-0.389626,-0.369714,0,1,0,0,0,0,BENIGN
2,-0.459864,-0.445500,0.037514,0.027864,0.193999,-0.004998,0.281243,-0.311710,0.006706,0.098573,...,-0.118364,-0.389626,-0.369714,0,1,0,0,0,0,BENIGN
3,1.378070,-0.481952,-0.011879,-0.009261,-0.058755,-0.007712,-0.311085,-0.311710,-0.327627,-0.265803,...,-0.118364,-0.389626,-0.369714,0,1,0,0,0,0,BENIGN
4,-0.459864,-0.445395,0.037514,0.026008,0.193999,-0.004998,0.281243,-0.311710,0.006706,0.098573,...,-0.118364,-0.389626,-0.369714,0,1,0,0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2427188,-0.437990,-0.481430,-0.011879,-0.010189,-0.058199,-0.007709,-0.303292,-0.214180,-0.297478,-0.265803,...,-0.118364,-0.389626,-0.369714,0,1,0,0,0,0,BENIGN
2427189,-0.458254,-0.480254,-0.010644,-0.009261,-0.051343,-0.007648,-0.259127,0.338491,-0.126635,-0.265803,...,-0.118364,-0.389626,-0.369714,0,1,0,0,0,0,BENIGN
2427190,-0.458254,-0.481950,-0.010644,-0.009261,-0.052825,-0.007672,-0.269518,0.208451,-0.166833,-0.265803,...,-0.118364,-0.389626,-0.369714,0,1,0,0,0,0,BENIGN
2427191,-0.458254,-0.481950,-0.010644,-0.009261,-0.051343,-0.007653,-0.259127,0.338491,-0.126635,-0.265803,...,-0.118364,-0.389626,-0.369714,0,1,0,0,0,0,BENIGN


In [41]:
## feature extraction ##

In [42]:
## feature extraction for multi class classification ##

In [43]:
numeric_multi = multi_data[numeric_col]
numeric_multi['intrusion'] = multi_data['intrusion']

/tmp/ipykernel_18351/597325382.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  numeric_multi['intrusion'] = multi_data['intrusion']


In [44]:
corr=numeric_multi.corr()
corr_y = abs(corr['intrusion'])
highest_corr = corr_y[corr_y >0.2]
highest_corr.sort_values(ascending=True)

FlowDuration            0.222434
FwdIATTotal             0.223412
FINFlagCount            0.226649
BwdPacketLengthMin      0.241209
MinPacketLength         0.253142
FlowIATStd              0.337680
IdleMin                 0.378827
FwdIATMax               0.388718
IdleMean                0.388999
FlowIATMax              0.389110
IdleMax                 0.392880
FwdIATStd               0.419182
PacketLengthMean        0.426136
AveragePacketSize       0.426443
PacketLengthVariance    0.446930
MaxPacketLength         0.458021
PacketLengthStd         0.473343
AvgBwdSegmentSize       0.490577
BwdPacketLengthMean     0.490577
BwdPacketLengthMax      0.493996
BwdPacketLengthStd      0.507965
intrusion               1.000000
Name: intrusion, dtype: float64

In [45]:
numeric_multi = multi_data[['FINFlagCount','FlowDuration','FwdIATTotal','BwdPacketLengthMin','MinPacketLength','AveragePacketSize','PacketLengthMean',
                           'PacketLengthVariance','FlowIATStd','MaxPacketLength','PacketLengthStd','AvgBwdSegmentSize','IdleMin',
                           'BwdPacketLengthMean','FlowIATMax','BwdPacketLengthMax','IdleMax','BwdPacketLengthStd','FwdIATMax',
                           'FwdIATStd','IdleMean']]

In [46]:

multi_data = numeric_multi.join(multi_data[['intrusion','BENIGN','Botnet', 'Dos', 'Firewall', 'PortScan','Label']])

In [47]:
multi_data.to_csv("multi_data_balanced.csv")
multi_data

,FINFlagCount,FlowDuration,FwdIATTotal,BwdPacketLengthMin,MinPacketLength,AveragePacketSize,PacketLengthMean,PacketLengthVariance,FlowIATStd,MaxPacketLength,...,FwdIATMax,FwdIATStd,IdleMean,intrusion,BENIGN,Botnet,Dos,Firewall,PortScan,Label
0,-0.184935,-0.446539,-0.437422,-0.606120,-0.653703,-0.302984,-0.263780,-0.288482,-0.384223,-0.059322,...,-0.364229,-0.354241,-0.384114,0,1,0,0,0,0,BENIGN
1,-0.184935,-0.445056,-0.435937,-0.606120,-0.653703,-0.302984,-0.263780,-0.288482,-0.384129,-0.059322,...,-0.364225,-0.354251,-0.384114,0,1,0,0,0,0,BENIGN
2,-0.184935,-0.445500,-0.436382,-0.606120,-0.653703,-0.304005,-0.264972,-0.287949,-0.383932,-0.059322,...,-0.364189,-0.354140,-0.384114,0,1,0,0,0,0,BENIGN
3,-0.184935,-0.481952,-0.472898,-0.606120,-0.653703,-0.626015,-0.610245,-0.320999,-0.396327,-0.512639,...,-0.402284,-0.369694,-0.384114,0,1,0,0,0,0,BENIGN
4,-0.184935,-0.445395,-0.436277,-0.606120,-0.653703,-0.296054,-0.256550,-0.287311,-0.383844,-0.059322,...,-0.364169,-0.354269,-0.384114,0,1,0,0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2427188,5.407292,-0.481430,-0.472898,-0.522001,-0.419333,-0.600322,-0.591658,-0.320999,-0.396329,-0.509853,...,-0.402284,-0.369694,-0.384114,0,1,0,0,0,0,BENIGN
2427189,-0.184935,-0.480254,-0.472898,0.487427,0.908763,-0.429033,-0.439239,-0.320754,-0.392246,-0.476411,...,-0.402284,-0.369694,-0.384114,0,1,0,0,0,0,BENIGN
2427190,-0.184935,-0.481950,-0.472898,0.066832,0.596270,-0.488984,-0.491284,-0.320956,-0.396323,-0.490345,...,-0.402284,-0.369694,-0.384114,0,1,0,0,0,0,BENIGN
2427191,-0.184935,-0.481950,-0.472898,0.403308,0.908763,-0.437597,-0.446674,-0.320825,-0.396319,-0.479198,...,-0.402284,-0.369694,-0.384114,0,1,0,0,0,0,BENIGN


In [48]:
# counting the attacks for multiclass

multi_data['intrusion'].value_counts()

0    2035505
2     320258
4      57316
3       8587
1       4061
Name: intrusion, dtype: int64

In [49]:

X = multi_data.iloc[:,0:16].to_numpy() # dataset excluding target attribute (encoded, one-hot-encoded,original)
y = multi_data['intrusion'] # target attribute


X_train, X_validation, y_train, y_validation = train_test_split(X, y, test_size=0.20, random_state=0)



In [50]:
X_train = preprocessing.scale(X_train)
X_train = preprocessing.normalize(X_train)

In [51]:
X_validation = preprocessing.scale(X_validation)
X_validation = preprocessing.normalize(X_validation)

In [52]:
print(len(X_train), "Training sequences",X_train.shape)
print(len(X_validation), "Validation sequences",X_validation.shape)

1940581 Training sequences (1940581, 16)
485146 Validation sequences (485146, 16)


In [53]:
X_train = np.reshape(X_train,(X_train.shape[0],X_train.shape[1],1))
X_validation = np.reshape(X_validation,(X_validation.shape[0],X_validation.shape[1],1))

In [54]:
X_train.shape

(1940581, 16, 1)

In [55]:
import time

Model = keras.Sequential([

        keras.layers.Conv2D(96,(4,4),input_shape=(X_train.shape[1],X_train.shape[2],1),activation='relu',padding='same'),
        keras.layers.Conv2D(64,(3,3),activation="relu",padding='same'),
        keras.layers.Conv2D(32,(2,2),activation="relu",padding='same'),
        keras.layers.Flatten(),
        keras.layers.Dense(512,activation="relu"),
        keras.layers.Dense(128,activation="relu"),
        keras.layers.Dense(32,activation="relu"),
        keras.layers.Dense(5,activation="softmax"),
    
    
    ])

Model.compile(optimizer='adam',loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
start_time = time.time()
#Training the model
Model.fit(X_train, y_train, epochs=5, batch_size=64) 
Model.summary()

# Final evaluation of the model
scores = Model.evaluate(X_validation, y_validation, verbose=0)
delta = time.time()- start_time
print("Accuracy: %.2f%%" % (scores[1]*100))
print("Training time: %.2f sec" % (delta))

2023-02-10 13:04:39.354324: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-02-10 13:04:40.484909: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14769 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:19:00.0, compute capability: 7.5
2023-02-10 13:04:40.485495: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 14769 MB memory:  -> device: 1, name: Quadro RTX 5000, pci bus id: 0000:1a:00.0, compute capability: 7.5
2023-02-10 13:04:40.485869: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/tas

Epoch 1/5


2023-02-10 13:04:42.144808: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8600
2023-02-10 13:04:42.621239: I tensorflow/compiler/xla/service/service.cc:173] XLA service 0x7fee5fae53c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2023-02-10 13:04:42.621264: I tensorflow/compiler/xla/service/service.cc:181]   StreamExecutor device (0): Quadro RTX 5000, Compute Capability 7.5
2023-02-10 13:04:42.621269: I tensorflow/compiler/xla/service/service.cc:181]   StreamExecutor device (1): Quadro RTX 5000, Compute Capability 7.5
2023-02-10 13:04:42.621273: I tensorflow/compiler/xla/service/service.cc:181]   StreamExecutor device (2): Quadro RTX 5000, Compute Capability 7.5
2023-02-10 13:04:42.625093: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2023-02-10 13:04:42.729124: I tensorflow/compiler/jit/xla_compi

30322/30322 [==============================] - 260s 8ms/step - loss: 0.0729 - sparse_categorical_accuracy: 0.9728
Epoch 2/5
30322/30322 [==============================] - 255s 8ms/step - loss: 0.0479 - sparse_categorical_accuracy: 0.9852
Epoch 3/5
30322/30322 [==============================] - 252s 8ms/step - loss: 0.0414 - sparse_categorical_accuracy: 0.9881
Epoch 4/5
30322/30322 [==============================] - 256s 8ms/step - loss: 0.0387 - sparse_categorical_accuracy: 0.9895
Epoch 5/5
30322/30322 [==============================] - 255s 8ms/step - loss: 0.0380 - sparse_categorical_accuracy: 0.9897
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 16, 1, 96)         1632      
                                                                 
 conv2d_1 (Conv2D)           (None, 16, 1, 64)         55360     
                                        

In [56]:
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error
                             ,mean_absolute_error,roc_auc_score,confusion_matrix)
from sklearn import metrics

In [59]:
y_pred = Model.predict(X_validation, verbose = 0)
# predict crisp classes for test set
yhat_classes = np.argmax(y_pred,axis=1)
# reduce to 1d array
yhat_probs = y_pred[:, 0]

 
# accuracy: (tp + tn) / (p + n)
accuracy = accuracy_score(y_validation, yhat_classes)
print('Accuracy: %f' % accuracy)
# precision tp / (tp + fp)
precision = precision_score(y_validation, yhat_classes,average='macro')
print('Precision: %f' % precision)
# recall: tp / (tp + fn)
recall = recall_score(y_validation, yhat_classes,average='macro')
print('Recall: %f' % recall)
# f1: 2 tp / (2 tp + fp + fn)
f1 = f1_score(y_validation, yhat_classes,average='macro')
print('F1 score: %f' % f1)


# ROC AUC
#auc = roc_auc_score(Y_validation, yhat_probs,multi_class='ovr')
#print('ROC AUC: %f' % auc)
# confusion matrix
matrix = confusion_matrix(y_validation, yhat_classes)
print(matrix)
# false alaram rate
false_alaram_rate = matrix[1,0]/(matrix[1,0]+matrix[0,0])
print('false alaram rate: %f' % false_alaram_rate)

Accuracy: 0.965932
Precision: 0.866508
Recall: 0.689423
F1 score: 0.730665
[[400434     18    315     27   6113]
 [   681    163      1      0      0]
 [  3379      0  61001      0      0]
 [   287      0      0   1343      0]
 [  5695      0     12      0   5677]]
false alaram rate: 0.001698
